# Buscador Semántico de Películas

**Trabajo Integrador — Procesamiento de Lenguaje Natural · Grupo 2**

Sistema de búsqueda semántica que recibe descripciones libres en lenguaje natural (por ejemplo: *"una película donde el protagonista pierde la memoria y hay viajes en el tiempo"*) y recupera las películas cuya información textual es semánticamente más similar a la consulta.

Se comparan dos enfoques de recuperación de información:

- **Baseline léxico:** TF-IDF + similitud coseno.
- **Enfoque semántico:** embeddings (Sentence-BERT) + similitud coseno.

Ambos enfoques se evalúan cuantitativamente con **Hit Rate@K** y **MRR** sobre un conjunto de consultas con película esperada (ground truth).

## 1. Configuración del entorno

In [1]:
# Dependencias (ejecutar en Google Colab si no están instaladas)
!pip install -q sentence-transformers scikit-learn


[notice] A new release of pip is available: 25.0.1 -> 26.1.1
[notice] To update, run: pip install --upgrade pip


In [2]:
import pandas as pd
import numpy as np

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sentence_transformers import SentenceTransformer

/Users/lucas/anaconda3/lib/python3.10/site-packages/sentence_transformers/cross_encoder/CrossEncoder.py:13: TqdmExperimentalWarning: Using `tqdm.autonotebook.tqdm` in notebook mode. Use `tqdm.tqdm` instead to force console mode (e.g. in jupyter console)
  from tqdm.autonotebook import tqdm, trange


## 2. Carga del dataset

El dataset contiene 1000 películas obtenidas de *The Movie Database* (TMDB), con título, sinopsis (`overview`), géneros, keywords, elenco (`cast`), director y metadatos numéricos.

In [3]:
import os

# Ruta relativa cuando se ejecuta desde la carpeta Notebook/ del repositorio.
# En Google Colab: subir el CSV al entorno y dejar solo el nombre del archivo.
CSV_PATH = "../data/movies_expanded_1000.csv"
if not os.path.exists(CSV_PATH):
    CSV_PATH = "movies_expanded_1000.csv"

df = pd.read_csv(CSV_PATH)
df.head()

,id,title,original_title,overview,tagline,release_date,runtime,original_language,genres,keywords,cast,directors,popularity,vote_average,vote_count,overview_length,release_year,search_text,search_text_length
0,10576,Psycho II,Psycho II,Norman Bates is declared sane and released fro...,"It's 22 years later, and Norman Bates is comin...",1983-06-03,113,en,"['Horror', 'Mystery', 'Thriller', 'Crime']","['psychopath', 'insanity', 'motel', 'sequel', ...","['Anthony Perkins', 'Meg Tilly', 'Vera Miles',...",['Richard Franklin'],2.7056,6.510,724,204,1983,Psycho II | Norman Bates is declared sane and ...,572
1,10830,Matilda,Matilda,Matilda Wormwood is an brilliant and intellige...,Somewhere inside all of us is the power to cha...,1996-08-02,98,en,"['Comedy', 'Family', 'Fantasy']","['based on novel or book', 'parent child relat...","['Mara Wilson', 'Danny DeVito', 'Rhea Perlman'...",['Danny DeVito'],11.4750,7.174,4785,413,1996,Matilda | Matilda Wormwood is an brilliant and...,1033
2,89,Indiana Jones and the Last Crusade,Indiana Jones and the Last Crusade,"In 1938, an art collector appeals to eminent a...",Have the adventure of your life keeping up wit...,1989-05-24,127,en,"['Adventure', 'Action']","['saving the world', 'nazi', 'holy grail', 've...","['Harrison Ford', 'Sean Connery', 'Denholm Ell...",['Steven Spielberg'],7.2998,7.852,11019,625,1989,"Indiana Jones and the Last Crusade | In 1938, ...",1328
3,12155,Alice in Wonderland,Alice in Wonderland,"Alice, now 19 years old, returns to the whimsi...",You're invited to a very important date.,2010-03-03,108,en,"['Family', 'Fantasy', 'Adventure']","['based on novel or book', 'queen', ""based on ...","['Mia Wasikowska', 'Johnny Depp', 'Anne Hathaw...",['Tim Burton'],10.4469,6.639,14744,139,2010,"Alice in Wonderland | Alice, now 19 years old,...",638
4,11186,Child's Play 2,Child's Play 2,Chucky is reconstructed by a toy factory to di...,Sorry Jack... Chucky's back!,1990-11-09,84,en,['Horror'],"['factory', 'voodoo', 'foster parents', 'faith...","['Alex Vincent', 'Brad Dourif', 'Christine Eli...",['John Lafia'],4.8559,6.359,2005,172,1990,Child's Play 2 | Chucky is reconstructed by a ...,638


## 3. Validación y limpieza

Verificamos dimensiones, valores nulos y duplicados antes de construir la representación textual.

In [4]:
print("Shape:", df.shape)

print("\nValores nulos por columna:")
print(df.isnull().sum())

print("\nDuplicados por id:", df.duplicated(subset="id").sum())
print("Duplicados por title:", df.duplicated(subset="title").sum())

Shape: (1000, 19)

Valores nulos por columna:
id                     0
title                  0
original_title         0
overview               0
tagline               86
release_date           0
runtime                0
original_language      0
genres                 0
keywords               0
cast                   0
directors              0
popularity             0
vote_average           0
vote_count             0
overview_length        0
release_year           0
search_text            0
search_text_length     0
dtype: int64

Duplicados por id: 0
Duplicados por title: 4


In [5]:
# Normalización de columnas textuales: nulos -> "", conversión a string y sin espacios laterales
text_cols = ["title", "overview", "genres", "keywords", "cast"]

for col in text_cols:
    df[col] = df[col].fillna("").astype(str).str.strip()

print("Columnas textuales normalizadas.")

Columnas textuales normalizadas.


## 4. Representación textual enriquecida

Para cada película construimos un único texto consolidado (`search_text`) que integra título, géneros, keywords, elenco y sinopsis en un formato estructurado y etiquetado. Esta representación es la entrada **tanto del baseline TF-IDF como del modelo de embeddings**, lo que garantiza una comparación justa entre ambos enfoques.

In [6]:
def build_search_text(row):
    return (
        f"Title: {row['title']}. "
        f"Genres: {row['genres']}. "
        f"Keywords: {row['keywords']}. "
        f"Cast: {row['cast']}. "
        f"Overview: {row['overview']}."
    )

df["search_text"] = df.apply(build_search_text, axis=1)
print(df.loc[0, "search_text"])

Title: Psycho II. Genres: ['Horror', 'Mystery', 'Thriller', 'Crime']. Keywords: ['psychopath', 'insanity', 'motel', 'sequel', 'revenge', 'murder', 'mental illness', 'framed for murder', 'voyeur', 'mother son relationship', 'admiring']. Cast: ['Anthony Perkins', 'Meg Tilly', 'Vera Miles', 'Robert Loggia', 'Dennis Franz']. Overview: Norman Bates is declared sane and released from the facility in which he was being held, despite the complaints of Lila Loomis, sister of his most famous victim. Is he really cured, or will he kill again?.


In [7]:
# Verificación de integridad de la representación
assert df["search_text"].isnull().sum() == 0
assert (df["search_text"].str.strip() == "").sum() == 0

movie_texts = df["search_text"].tolist()
print("Películas a vectorizar:", len(movie_texts))

Películas a vectorizar: 1000


## 5. Baseline léxico: TF-IDF + similitud coseno

El baseline representa cada película mediante un vector TF-IDF (frecuencia de término ponderada por frecuencia inversa de documento) y recupera resultados por similitud coseno frente al vector de la consulta. Captura coincidencias **léxicas**, no semánticas.

In [8]:
tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english",
    max_features=5000,
)

tfidf_matrix = tfidf_vectorizer.fit_transform(movie_texts)
print("Shape matriz TF-IDF:", tfidf_matrix.shape)

Shape matriz TF-IDF: (1000, 5000)


In [9]:
def search_tfidf(query, top_k=5):
    query_vec = tfidf_vectorizer.transform([query])
    similarities = cosine_similarity(query_vec, tfidf_matrix)[0]
    top_idx = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_idx][["title", "overview", "genres", "keywords", "cast"]].copy()
    results["similarity"] = similarities[top_idx]
    return results.reset_index(drop=True)

In [10]:
search_tfidf("movie about dreams, mind and memory", top_k=5)

,title,overview,genres,keywords,cast,similarity
0,Dark City,"A man struggles with memories of his past, inc...","['Mystery', 'Science Fiction']","['beach', 'amnesia', 'experiment', 'paranoia',...","['Rufus Sewell', 'William Hurt', 'Kiefer Suthe...",0.206517
1,In Your Dreams,Stevie and her little brother Elliot journey i...,"['Adventure', 'Animation', 'Comedy', 'Family',...","['magic', 'wish', 'family', 'dream']","['Jolie Hoang-Rappaport', 'Elias Janssen', 'Si...",0.167585
2,The Maze Runner,A teenager with no memory of his past finds hi...,"['Action', 'Mystery', 'Science Fiction', 'Thri...","['based on novel or book', 'escape', 'dystopia...","[""Dylan O'Brien"", 'Kaya Scodelario', 'Thomas B...",0.136834
3,Lola's Secret,Young man has his dreams come true when the se...,"['Drama', 'Comedy', 'Horror']","['sexploitation', 'maid', 'seductress', 'voyeu...","['Donatella Damiani', 'Scott Coffey', 'Gabriel...",0.134166
4,50 First Dates,Henry is a player skilled at seducing women. B...,"['Comedy', 'Romance']","['amnesia', 'hawaii', 'romcom', 'deception', '...","['Adam Sandler', 'Drew Barrymore', 'Rob Schnei...",0.129778


## 6. Enfoque semántico: embeddings (Sentence-BERT)

Usamos el modelo preentrenado `all-MiniLM-L6-v2`, liviano y eficiente, para convertir cada texto en un vector denso de **384 dimensiones** que captura su significado. La recuperación se realiza por similitud coseno entre el embedding de la consulta y los de las películas.

In [11]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

/Users/lucas/anaconda3/lib/python3.10/site-packages/transformers/tokenization_utils_base.py:1617: FutureWarning: `clean_up_tokenization_spaces` was not set. It will be set to `True` by default. This behavior will be deprecated in transformers v4.45, and will be then set to `False` by default. For more details check this issue: https://github.com/huggingface/transformers/issues/31884
  warnings.warn(


In [12]:
movie_embeddings = model.encode(
    movie_texts,
    convert_to_numpy=True,
    show_progress_bar=True,
)
print("Shape de la matriz de embeddings:", movie_embeddings.shape)

Batches:   0%|          | 0/32 [00:00<?, ?it/s]

Shape de la matriz de embeddings: (1000, 384)


In [13]:
def search_semantic(query, top_k=5):
    query_embedding = model.encode([query], convert_to_numpy=True)
    similarities = cosine_similarity(query_embedding, movie_embeddings)[0]
    top_idx = np.argsort(similarities)[::-1][:top_k]

    results = df.iloc[top_idx][["title", "overview", "genres", "keywords", "cast"]].copy()
    results["similarity"] = similarities[top_idx]
    return results.reset_index(drop=True)

## 7. Evaluación cualitativa

Inspeccionamos los resultados del buscador semántico sobre consultas en lenguaje natural para verificar la coherencia de las películas recuperadas.

In [14]:
queries = [
    "animated movie with dinosaurs and friendship",
    "historical drama about royalty and power",
    "film about genius, science and war",
    "movie about dreams, memory and the mind",
]

for q in queries:
    print("=" * 100)
    print("Consulta:", q)
    display(search_semantic(q, top_k=3))

Consulta: animated movie with dinosaurs and friendship


,title,overview,genres,keywords,cast,similarity
0,The Good Dinosaur,An epic journey into the world of dinosaurs wh...,"['Adventure', 'Animation', 'Family']","['friendship', 'tyrannosaurus rex', 'cartoon',...","['Frances McDormand', 'Raymond Ochoa', 'Jeffre...",0.657111
1,The Land Before Time,An orphaned brontosaurus named Littlefoot sets...,"['Family', 'Animation', 'Adventure']","['vulkan', 'loss of loved one', 'tyrannosaurus...","['Gabriel Damon', 'Candace Hutson', 'Will Ryan...",0.510797
2,Madagascar,Four animal friends get a taste of the wild li...,"['Adventure', 'Animation', 'Comedy', 'Family']","['friendship', 'island', 'escape', 'africa', '...","['Ben Stiller', 'Chris Rock', 'David Schwimmer...",0.501774


Consulta: historical drama about royalty and power


,title,overview,genres,keywords,cast,similarity
0,The Prince of Egypt,The strong bond between two Royal Egyptian bro...,"['Adventure', 'Animation', 'Drama', 'Family']","['egypt', 'moses', 'kingdom', 'pyramid', 'exod...","['Val Kilmer', 'Ralph Fiennes', 'Michelle Pfei...",0.562820
1,"Red, White & Royal Blue","After an altercation between Alex, the preside...","['Comedy', 'Romance']","['based on novel or book', 'politics', 'prince...","['Taylor Zakhar Perez', 'Nicholas Galitzine', ...",0.525003
2,The Last Kingdom: Seven Kings Must Die,"In the wake of King Edward's death, Uhtred of ...","['Action', 'Adventure', 'War']","['fight', 'kingdom', 'heir to the throne', 'ba...","['Alexander Dreymon', 'Harry Gilby', 'Mark Row...",0.514684


Consulta: film about genius, science and war


,title,overview,genres,keywords,cast,similarity
0,War Machine,A rock star general bent on winning the “impos...,"['Comedy', 'Drama', 'War']","['journalist', 'based on novel or book', 'afgh...","['Brad Pitt', 'Anthony Michael Hall', 'Emory C...",0.491720
1,Oppenheimer,The story of J. Robert Oppenheimer's role in t...,"['Drama', 'History']","['husband wife relationship', 'based on novel ...","['Cillian Murphy', 'Emily Blunt', 'Matt Damon'...",0.476416
2,The Sum of All Fears,"When the president of Russia suddenly dies, a ...","['Thriller', 'Action', 'Drama']","['central intelligence agency (cia)', 'usa pre...","['Ben Affleck', 'Morgan Freeman', 'James Cromw...",0.454253


Consulta: movie about dreams, memory and the mind


,title,overview,genres,keywords,cast,similarity
0,50 First Dates,Henry is a player skilled at seducing women. B...,"['Comedy', 'Romance']","['amnesia', 'hawaii', 'romcom', 'deception', '...","['Adam Sandler', 'Drew Barrymore', 'Rob Schnei...",0.471514
1,Inception,"Cobb, a skilled thief who commits corporate es...","['Action', 'Science Fiction', 'Adventure']","['mission', 'dreams', 'kidnapping', 'spy', 'al...","['Leonardo DiCaprio', 'Joseph Gordon-Levitt', ...",0.445464
2,Paprika,When a machine that allows therapists to enter...,"['Science Fiction', 'Thriller', 'Animation']","['research', 'japan', 'dreams', 'based on nove...","['Megumi Hayashibara', 'Tohru Emori', 'Katsuno...",0.439657


## 8. Evaluación cuantitativa: Hit Rate@K y MRR

Definimos un conjunto de consultas sintéticas, cada una con su película esperada (*ground truth*), diseñadas para simular búsquedas reales basadas en paráfrasis, sinónimos y descripciones parciales. Medimos:

- **Hit Rate@K:** proporción de consultas cuya película esperada aparece dentro del Top-K (métrica binaria por consulta).
- **MRR (Mean Reciprocal Rank):** promedio del inverso de la posición del resultado correcto (1 si aparece primero, 0.5 si segundo, etc.), midiendo la precisión posicional del sistema.

In [15]:
# Conjunto de evaluación: (consulta sintética, título esperado)
evaluation_queries = [
    ("child prodigy with telekinesis directed by Danny DeVito", "Matilda"),
    ("Harrison Ford and Sean Connery search for the Holy Grail", "Indiana Jones and the Last Crusade"),
    ("horror sequel possessed doll in toy factory", "Child's Play 2"),
    ("alien time loop Tom Cruise", "Edge of Tomorrow"),
    ("James Bond playing poker in Montenegro Daniel Craig", "Casino Royale"),
    ("Vietnam veteran with low IQ Tom Hanks", "Forrest Gump"),
    ("Jason Momoa and Jack Black trapped in a cubic world", "A Minecraft Movie"),
    ("boy reads a magical book about a luck dragon", "The NeverEnding Story"),
    ("stuntman searches for missing movie star Ryan Gosling", "The Fall Guy"),
    ("Al Pacino seeks forgiveness as Michael Corleone", "The Godfather Part III"),
    ("mutant mercenary who breaks the fourth wall Ryan Reynolds", "Deadpool 2"),
    ("monsters from another dimension destroy the world 2025 Milla Jovovich", "Worldbreaker"),
    ("Cillian Murphy atomic bomb Nolan", "Oppenheimer"),
    ("Peter Parker multiverse Doctor Strange", "Spider-Man: No Way Home"),
    ("thief who infiltrates dreams DiCaprio", "Inception"),
    ("cyberpunk sequel Keanu Reeves the One", "The Matrix Reloaded"),
    ("Brad Pitt and Morgan Freeman 7 deadly sins killer", "Se7en"),
    ("black and white movie Liam Neeson saving Jews", "Schindler's List"),
    ("freed slave western looking for his wife Jamie Foxx", "Django Unchained"),
    ("Uma Thurman assassin with samurai sword revenge", "Kill Bill: Vol. 1"),
    ("Irish mob infiltrators in Boston DiCaprio Matt Damon", "The Departed"),
    ("Christian Bale vs Heath Ledger as the Joker", "The Dark Knight"),
    ("romance on a ship 1912 DiCaprio and Kate Winslet", "Titanic"),
    ("lion exiled by his uncle Scar", "The Lion King"),
    ("Matthew McConaughey travels through a wormhole", "Interstellar"),
    ("New York ballerina obsessed with perfection Natalie Portman", "Black Swan"),
    ("ogre travels to Far Far Away to meet his in-laws", "Shrek 2"),
    ("arrogant race car gets lost on Route 66", "Cars"),
    ("Disney lion reclaims throne after Mufasa's death", "The Lion King"),
    ("pet chameleon becomes sheriff Johnny Depp", "Rango"),
    ("alien supervillain creates a new hero Will Ferrell", "Megamind"),
    ("ordinary Lego figure has to save the universe", "The Lego Movie"),
    ("Madrigal family in Colombia girl without magic", "Encanto"),
    ("emotions get lost in the mind of an 11-year-old girl", "Inside Out"),
    ("stop-motion thief fox against three farmers Wes Anderson", "Fantastic Mr. Fox"),
    ("Miles Morales teams up with other Spider-Men from different dimensions", "Spider-Man: Into the Spider-Verse"),
    ("old man who flies his house with balloons and a boy scout", "Up"),
    ("clownfish looking for his son in a dentist's aquarium", "Finding Nemo"),
    ("young Chinese woman disguises herself as a soldier guardian dragon", "Mulan"),
    ("bunny cop and con artist fox investigate mystery", "Zootopia"),
    ("supervillain adopts three girls to steal the moon", "Despicable Me"),
    ("time loop fighting aliens", "Edge of Tomorrow"),
    ("little girl telekinesis", "Matilda"),
    ("finding the holy grail", "Indiana Jones and the Last Crusade"),
    ("killer doll toy factory", "Child's Play 2"),
    ("mafia part 3", "The Godfather Part III"),
    ("ballet dancer thriller", "Black Swan"),
    ("horror cant make noise", "A Quiet Place"),
    ("skeleton pirates curse", "Pirates of the Caribbean: The Curse of the Black Pearl"),
    ("spider multiverse", "Spider-Man: No Way Home"),
    ("red race car route 66", "Cars"),
    ("ogre sequel in laws", "Shrek 2"),
    ("ship sink romance", "Titanic"),
    ("stealing dreams heist", "Inception"),
    ("saving jews ww2", "Schindler's List"),
    ("matrix sequel zion", "The Matrix Reloaded"),
]

print("Consultas de evaluación:", len(evaluation_queries))

Consultas de evaluación: 56


In [16]:
def evaluate_system(search_function, queries, top_k=5):
    """Evalúa un sistema de búsqueda devolviendo Hit Rate@K y MRR."""
    hits = 0
    reciprocal_ranks = []

    for query, expected_title in queries:
        retrieved_titles = search_function(query, top_k=top_k)["title"].tolist()

        if expected_title in retrieved_titles:
            hits += 1
            rank = retrieved_titles.index(expected_title) + 1
            reciprocal_ranks.append(1.0 / rank)
        else:
            reciprocal_ranks.append(0.0)

    hit_rate = hits / len(queries)
    mrr = np.mean(reciprocal_ranks)
    return hit_rate, mrr

In [17]:
print("Evaluando TF-IDF (léxico)...")
tfidf_hit, tfidf_mrr = evaluate_system(search_tfidf, evaluation_queries, top_k=5)

print("Evaluando Embeddings (semántico)...")
sem_hit, sem_mrr = evaluate_system(search_semantic, evaluation_queries, top_k=5)

comparison_df = pd.DataFrame({
    "Modelo": ["TF-IDF (Léxico)", "Embeddings (Semántico)"],
    "Hit Rate@5": [f"{tfidf_hit:.2%}", f"{sem_hit:.2%}"],
    "MRR": [round(tfidf_mrr, 4), round(sem_mrr, 4)],
})

print("\n--- Resultados de la Evaluación ---")
comparison_df

Evaluando TF-IDF (léxico)...
Evaluando Embeddings (semántico)...

--- Resultados de la Evaluación ---


,Modelo,Hit Rate@5,MRR
0,TF-IDF (Léxico),87.50%,0.8214
1,Embeddings (Semántico),94.64%,0.8405


## 9. Conclusiones

- El enfoque **TF-IDF** depende de coincidencias léxicas exactas: cuando la consulta usa sinónimos o paráfrasis conceptuales en lugar de los términos presentes en la descripción, su capacidad para posicionar la película esperada se degrada.
- El enfoque basado en **embeddings (Sentence-BERT)** captura el significado abstracto de la consulta, logrando mejor **Hit Rate@K** y **MRR**, y resultando un buscador más robusto y natural para el usuario final.
- La representación textual enriquecida (título + géneros + keywords + elenco + sinopsis) aporta contexto semántico que beneficia especialmente al modelo de embeddings.

## 9. Conclusiones

La evaluación cuantitativa sobre las 56 consultas con película esperada (*ground truth*) confirma la ventaja del enfoque semántico:

| Modelo | Hit Rate@5 | MRR |
|---|---|---|
| TF-IDF (léxico) | 87.50% | 0.8214 |
| Embeddings (semántico) | **94.64%** | **0.8405** |

- El enfoque **TF-IDF** depende de coincidencias léxicas exactas: cuando la consulta usa sinónimos o paráfrasis conceptuales en lugar de los términos presentes en la descripción, su capacidad para posicionar la película esperada se degrada.
- El enfoque basado en **embeddings (Sentence-BERT)** captura el significado abstracto de la consulta, mejorando el Hit Rate@5 en **+7.14 puntos porcentuales** y obteniendo un MRR más alto: no solo recupera la película correcta con mayor frecuencia, sino que la ubica en mejores posiciones del ranking.
- La evaluación cualitativa es coherente con la cuantitativa: consultas descriptivas como *"film about genius, science and war"* o *"movie about dreams, memory and the mind"* recuperan películas pertinentes (Oppenheimer, Inception, Paprika) pese a no compartir las palabras exactas de la sinopsis.
- **Limitaciones:** el dataset se limita a 1000 películas, el modelo `all-MiniLM-L6-v2` está optimizado para inglés, y algunas consultas del set de evaluación referencian títulos ausentes del dataset (lo que penaliza por igual a ambos enfoques).
- **Trabajo futuro:** incorporar un modelo multilingüe, ampliar el dataset vía la API de TMDB, usar un índice vectorial para escalar la búsqueda y exponer el sistema mediante un frontend interactivo.